# Download Data From Kaggle

For more information about the competition check the [link](https://www.kaggle.com/competitions/titanic/)

In [66]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running on Google Colab. Setting up Kaggle data...")
    from google.colab import userdata
    import json

    # Setup Kaggle API
    os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
    os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

    !mkdir -p ~/.kaggle
    kaggle_config = {"username": os.environ["KAGGLE_USERNAME"], "key": os.environ["KAGGLE_KEY"]}
    with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
        json.dump(kaggle_config, f)
    !chmod 600 ~/.kaggle/kaggle.json

    # Download data if it doesn't exist yet
    if not os.path.exists('data'):
        !kaggle competitions download -c titanic
        !unzip -q titanic.zip -d data/
        !rm titanic.zip
        print("Data downloaded and unzipped.")
else:
    if not os.path.exists('data'):
        print("Download data ...")
        !kaggle competitions download -c titanic
        !unzip -q titanic.zip -d data/
        !rm titanic.zip
        print("Data downloaded")
    else:
        print('Data already exists')

Running on Google Colab. Setting up Kaggle data...


# Exploring The Dataset

In [67]:
import pandas as pd

df = pd.read_csv('data/train.csv')

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### Missing values

In [68]:
print("Missing values in the dataset:\n"+"="*20)
print(df.isna().sum())
print("="*20)
df[df.isna().any(axis=1)]

Missing values in the dataset:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
884,885,0,3,"Sutehall, Mr. Henry Jr",male,25.0,0,0,SOTON/OQ 392076,7.0500,NaN,S
885,886,0,3,"Rice, Mrs. William (Margaret Norton)",female,39.0,0,5,382652,29.1250,NaN,Q
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S


### Filling mising values

In [69]:
mean_age = df['Age'].mean()
df['Age'] = df['Age'].fillna(mean_age)
df['Embarked'] = df['Embarked'].fillna('S')
df['Cabin'] = df['Cabin'].ffill()
df.isna().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


## Statistics

In [70]:
total_passengers = df.shape[0]
survived_passengers = df[df["Survived"] == 1].shape[0]

print(f"total number of passengers from southampton: {df[df['Embarked'] == 'S'].shape[0]}")
print(f"total number of passengers from cherbourg: {df[df['Embarked'] == 'C'].shape[0]}")
print(f"total number of passengers from queenstown: {df[df['Embarked'] == 'Q'].shape[0]}")

total number of passengers from southampton: 646
total number of passengers from cherbourg: 168
total number of passengers from queenstown: 77


## Remove unsed columns

In [71]:
df.drop(['Embarked', 'Cabin', 'Ticket', 'Name', 'PassengerId', 'Fare'], axis=1, inplace=True)
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch
0,0,3,male,22.0,1,0
1,1,1,female,38.0,1,0
2,1,3,female,26.0,0,0
3,1,1,female,35.0,1,0
4,0,3,male,35.0,0,0


In [72]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['Sex'] = le.fit_transform(df['Sex'])

In [73]:
Y = df['Survived']
X = df.drop('Survived', axis=1)

In [74]:
Y = df['Survived']
X = df.drop(["Survived"], axis=1)

In [76]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from scipy.stats import uniform, randint

pipeline = Pipeline(
    [
        ('scaller', StandardScaler()),
        ("clf", SVC()),
    ]
)

param_distribution = [
    {
        "scaller": [StandardScaler()],
        "clf": [SVC()],
        "clf__C": uniform(0.1, 2).rvs(150),
        "clf__kernel": ["poly", "linear", "rbf"],
        "clf__degree": [2, 3, 4, 5, 6, 7],
        "clf__class_weight": [True, False]
    },
]

search = RandomizedSearchCV(estimator=pipeline, param_distributions=param_distribution, verbose=3, n_iter=100, cv=KFold(n_splits=5))

search.fit(X, Y)
print(f"best_score: {search.best_score_}")
print(f"best_params: {search.best_params_}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV 1/5] END clf=SVC(), clf__C=0.24749742126306917, clf__degree=7, clf__kernel=linear, scaller=StandardScaler();, score=0.816 total time=   0.1s
[CV 2/5] END clf=SVC(), clf__C=0.24749742126306917, clf__degree=7, clf__kernel=linear, scaller=StandardScaler();, score=0.798 total time=   0.0s
[CV 3/5] END clf=SVC(), clf__C=0.24749742126306917, clf__degree=7, clf__kernel=linear, scaller=StandardScaler();, score=0.781 total time=   0.0s
[CV 4/5] END clf=SVC(), clf__C=0.24749742126306917, clf__degree=7, clf__kernel=linear, scaller=StandardScaler();, score=0.736 total time=   0.0s
[CV 5/5] END clf=SVC(), clf__C=0.24749742126306917, clf__degree=7, clf__kernel=linear, scaller=StandardScaler();, score=0.803 total time=   0.0s
[CV 1/5] END clf=SVC(), clf__C=1.0503726242747093, clf__degree=3, clf__kernel=linear, scaller=StandardScaler();, score=0.816 total time=   0.0s
[CV 2/5] END clf=SVC(), clf__C=1.0503726242747093, clf__degree=3, clf